# ROGII — Colab runner (idempotent)

A **thin, stable launcher.** It only mounts Drive, sets Kaggle auth, and fresh-clones the
repo — then runs `colab/bootstrap.py`, which holds ALL the run logic (install, data, run,
Drive-sync) and is pulled fresh every time. **You never edit this notebook.**

- **Run an experiment:** Runtime → Run all.
- **Switch experiment / change the flow:** edit `SPRINT_ACTIVE.txt` (or `colab/bootstrap.py`)
  in the repo, push, then re-run the **Run** cell — it re-clones and picks up the latest.
- **Prereqs:** a Colab Secret `KAGGLE_API_TOKEN` (your 37-char token, notebook access ON).
  kaggle 2.x needs no username. Drive holds artifacts (probs/submissions/experiments.jsonl).

## 1. Setup — Drive + Kaggle auth (stable; rarely changes)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['DRIVE_ROOT'] = '/content/drive/MyDrive/Colab Notebooks/kaggle/rogii'        # bootstrap syncs artifacts here
os.makedirs(os.environ['DRIVE_ROOT'], exist_ok=True)

# Secrets → env (🔑 icon, notebook access ON). bootstrap.py inherits these.
#   KAGGLE_API_TOKEN — required (kaggle 2.x bearer token, no username needed)
#   GH_TOKEN         — optional (fine-grained PAT, Contents:RW on this repo) to persist the diary to git
from google.colab import userdata
for name in ('KAGGLE_API_TOKEN', 'GH_TOKEN'):
    try:
        v = userdata.get(name)
        if v:
            os.environ[name] = v
            print(f'✓ {name} set')
        else:
            print(f'⚠ {name} secret is empty')
    except Exception:
        opt = ' (optional — diary won\'t persist to git)' if name == 'GH_TOKEN' else ''
        print(f'⚠ no {name} secret{opt} — add via 🔑, notebook access ON')


## 2. Run — fresh clone + bootstrap (all logic lives in the repo)

In [ ]:
%cd /content
!rm -rf rogii-wellbore-geology-prediction
!git clone -q https://github.com/SirGrigor/rogii-wellbore-geology-prediction.git
%cd rogii-wellbore-geology-prediction
!git log -1 --oneline
# bootstrap.py inherits DRIVE_ROOT + KAGGLE_API_TOKEN from the env set above.
!python colab/bootstrap.py


## 3. (optional) Download the newest submission to local

Artifacts also sync to Drive automatically. For the kernels-only final submission you run `notebooks/kaggle_infer.ipynb` on Kaggle (see docs/submission_workflow.md) — not this.

In [ ]:
from pathlib import Path
from google.colab import files
subs = sorted(Path('/content/rogii-wellbore-geology-prediction/submissions').glob('*.csv'), key=lambda p: -p.stat().st_mtime)
if subs:
    print('newest:', subs[0].name); files.download(str(subs[0]))
else:
    print('no submissions yet')
